# [실습] Agent 구현

LangChain의 `create_agent`를 사용하여 가장 간단한 형태의 에이전트를 구현합니다.

## 학습 목표

- `create_agent`로 ReAct 에이전트를 구현하는 방법 학습
- TavilySearch + 다양한 커스텀 도구를 에이전트에 연결
- System Prompt 변경에 따른 에이전트 행동 변화 실험
- Checkpointer를 활용한 멀티턴 대화 구현

## 환경 설정

In [ ]:
%pip install tzdata dotenv rich langchain langchain-openai langchain-tavily langgraph -q

In [ ]:
from dotenv import load_dotenv
import os
from langchain.messages import HumanMessage,AIMessage,SystemMessage

load_dotenv('.env', override=True)

if os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키 설정 완료!')


---
## 1. ReAct Agent란?

ReAct(Reasoning + Acting)는 LLM이 생각(Reasoning)과 행동(Acting)을 번갈아 수행하는 패턴입니다.

```
사용자 질문
  → LLM이 어떤 도구를 사용할지 판단 (Reasoning)
  → 도구 실행 (Acting)
  → LLM이 결과를 분석하고 추가 행동 결정 (Reasoning)
  → (반복)
  → 최종 응답 생성
```

---
## 2. 도구(Tool) 불러오기

### 2-1. 검색 도구와 API 키

`web_search`는 Tavily(https://app.tavily.com) 검색 API를 사용합니다.   

Tavily Search는 월 1,000회의 무료 검색을 지원합니다.

`.env` 파일에 `TAVILY_API_KEY='tvly-...'`을 추가하세요.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv('.env', override=True)

if os.environ.get('TAVILY_API_KEY'):
    print('Tavily API 키 설정 완료!')
else:
    print('TAVILY_API_KEY가 없습니다. web_search는 대체 도구로 동작합니다.')

### 2-2. 표준 도구 불러오기

`tools.py`를 통해, 자주 사용할 도구를 불러오겠습니다.


In [ ]:
from tools import (
    DEFAULT_TOOLS,
    web_search,
    calculator,
    get_current_datetime,
    text_analyzer,
    save_note,
    get_notes,
)

tools = DEFAULT_TOOLS
print(f'등록된 도구 ({len(tools)}개):')
for t in tools:
    print(f'  - {t.name}')

In [ ]:
from rich import print as rprint

for t in tools:
    print(f'  - {t.name}\n{t.description}\n')
    rprint(t.args)

---
## 3. create_agent로 ReAct Agent 만들기

LangChain 1.x의 `create_agent`는 내부적으로 LangGraph 기반의 그래프를 생성하며, 도구 호출 루프를 자동으로 처리합니다.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    'gpt-5.2',
    tools=tools,
    system_prompt='도구를 실행하기 전, 중간 과정을 간략히 설명하세요.'
)

# langgraph.prebuilt.create_react_agent는 Deprecated
# 바이브 코딩 시 참고해 주세요!

print(f'Agent 타입: {type(agent).__name__}')
agent

### 3-1. Agent 실행

Agent는 Tool 호출 + Tool 결과 + 다음 작업의 루프를 반복하며,    
HumanMessage --> AIMessage --> ToolMessage --> AIMessage --> ... 의 컨텍스트를 저장합니다.   

LLM과 동일하게 `invoke()`로 실행하며, 입력은 `messages` 키를 포함하는 dict입니다.

In [ ]:
result = agent.invoke({
    'messages': [HumanMessage('안녕하세요? 선릉역 멀티캠퍼스 근처 저녁식사로 괜찮은 맛집 추천해 주세요. 네이버 링크도 공유해주세요.')]
})

# 최종 응답
rprint(result['messages'][-1].text)

### 3-2. 메시지 흐름 분석

에이전트가 동작한 전체 과정을 메시지 흐름으로 확인할 수 있습니다.

In [ ]:
def print_message_flow(messages):
    """에이전트의 메시지 흐름을 보기 좋게 출력"""
    for i, msg in enumerate(messages):
        msg_type = type(msg).__name__
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            calls = ', '.join([f"{tc['name']}({tc['args']})" for tc in msg.tool_calls])
            print(f'  [{i}] {msg_type}: tool_calls=[{calls}]')
        elif hasattr(msg, 'tool_call_id'):
            preview = msg.text[:100] + '...' if len(msg.text) > 100 else msg.text
            print(f'  [{i}] {msg_type} ({msg.name}): {preview}')
        else:
            preview = msg.text  if len(msg.text) > 100 else msg.text
            print(f'  [{i}] {msg_type}: {preview}')

print_message_flow(result['messages'])

## 스트리밍 헬퍼 라이브러리
전체를 스트리밍하여 아래와 같이 구성할 수도 있습니다.

In [ ]:
from stream_utils import stream_print, stream_with_markdown

result = await stream_print(agent, '오늘 인공지능 관련 주요 뉴스 알려줘요')

In [ ]:
# 마크다운 자동 렌더링
result = await stream_with_markdown(agent, '''오늘 프로야구 선발투수 매치업 알려줘. 5경기 10명이야.
                                    한국 소식은 topic을 general로 해줘.
                                    오늘날짜를 검색어에 포함해줘''')

---
## 4. 다양한 질문으로 Agent 실행하기

에이전트는 질문의 유형에 따라 적절한 도구를 선택하여 사용합니다.

### 4-1. 검색이 필요한 질문

In [ ]:
result = await stream_with_markdown(agent, 'Anthropic이 만든, LLM Agent의 "Skills"에 대해 설명해줘. MCP와 뭐가 달라? Skill/MCP 표준 스펙을 바탕으로 설명해줘.')

### 4-2. 계산이 필요한 질문

In [ ]:
result = await stream_with_markdown(agent, '2의 20승에 3의 30승을 곱하면 얼마인가요?')

### 4-3. 텍스트 분석 질문

In [ ]:
result = await stream_with_markdown(agent, '다음 문장의 글자 수와 단어 수를 분석해줘: "LangGraph는 LLM 기반의 복잡한 워크플로우를 그래프 구조로 정의하는 프레임워크입니다."')

### 4-4. 메모 저장 및 조회

In [ ]:
# 메모 저장
result = agent.invoke({
    'messages': [HumanMessage("오늘 할 일을 메모해줘. 제목은 '오늘 할 일'이고 내용은 'LangGraph 실습 완료하기'야.")]
})
print(f'[저장] {result["messages"][-1].text}')

In [ ]:
# 메모 조회
result = agent.invoke({
    'messages': [HumanMessage('저장된 메모를 보여줘.')]
})
print(f'[조회] {result["messages"][-1].text}')

### 4-5. 여러 도구를 조합하는 질문

In [ ]:
result = agent.invoke({
    'messages': [HumanMessage('오늘 날짜 알려주고, 올해가 며칠이 지났는지도 계산해줘.')]
})
print(result['messages'][-1].text)
print('\n--- 메시지 흐름 ---')
print_message_flow(result['messages'])

---
## Agent의 구조화 출력

Agent 단계에서도, 출력의 포맷을 구조화할 수 있습니다.    
create_agent의 `response_format`은 Pydantic 스키마를 받습니다.   

Agent 단계의 response format은 LLM에 비해 더 유연한 장점을 갖습니다.    
만약 파싱이 깨지는 경우, Tool 결과로 오류 메시지를 반환하여, 모델이 이를 수정할 수 있도록 합니다.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class SearchResult(BaseModel):
    """사용자의 질문에 대한 조사 결과"""
    summary: str = Field(description='한 문장 답변 요약')
    category: Literal['web_search_needed', 'web_search_not_needed'] = Field(
        description='''인터넷 검색이 필요했던 질문은 web_search_needed로 표시,
LLM 지식만으로 답할 수 있는 질문은 'web_search_not_needed'로 표시''')
    references: list[str] = Field(description='관련 참고 링크 리스트')
    
SYSTEM_PROMPT = """도구를 실행하기 전, 중간 과정을 간략히 설명하세요."""

search_agent = create_agent(
    'gpt-5.2',
    tools=[web_search],
    system_prompt=SYSTEM_PROMPT,
    response_format=SearchResult,
)

In [ ]:
questions = [
    '2025년 노벨 물리학상 수상자가 누구였는지 찾아줘',
]

for q in questions:
    result = search_agent.invoke({'messages': [HumanMessage(q)]})['structured_response']

result

In [ ]:
from rich import print as rprint

rprint(result)

---
## 5. 메모리(Checkpointer)를 사용한 멀티턴 대화

`InMemorySaver`를 사용하면 에이전트가 이전 대화를 기억하고 맥락을 유지할 수 있습니다.

`thread_id`로 서로 다른 대화 세션을 구분합니다.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agent_with_memory = create_agent(
    'gpt-5.2',
    tools=tools,
    system_prompt='도구를 실행하기 전, 중간 과정을 간략히 설명하세요.',
    checkpointer=InMemorySaver()
)

# 동일한 thread_id로 대화 연속
config = {'configurable': {'thread_id': 'session-1'}}

In [ ]:
# 첫 번째 대화
r1 = agent_with_memory.invoke(
    {'messages': [{'role': 'user', 'content': '안녕! 내 이름은 철수야.'}]},
    config=config
)
print(f'[턴 1] {r1["messages"][-1].text}')

In [ ]:
# 두 번째 대화 - 이전 맥락 기억 확인
r2 = agent_with_memory.invoke(
    {'messages': [{'role': 'user', 'content': '내 이름이 뭐라고 했지?'}]},
    config=config
)
print(f'[턴 2] {r2["messages"][-1].text}')

In [ ]:
r3 = await stream_with_markdown(agent_with_memory, '내 이름으로 삼행시 지어봐', config=config)

In [ ]:
# 다른 세션은 이전 대화를 모름
config_new = {'configurable': {'thread_id': 'session-2'}}

r3 = agent_with_memory.invoke(
    {'messages': [{'role': 'user', 'content': '내 이름이 뭐라고 했지?'}]},
    config=config_new
)
print(f'[새 세션] {r3["messages"][-1].text}')

---
## 6. build_agent로 에이전트 스크립트 저장하기

지금까지 배운 에이전트를 배포할 수 있도록, 에이전트를 정의하는 스크립트를 구성합니다.

In [ ]:
%%writefile build_agent.py
"""build_agent.py

에이전트를 한 곳에서 조립하는 팩토리입니다. 실습이 진행되며 도구, 미들웨어,
스킬, 메모리가 이 파일에 누적됩니다. 서비스 코드(예: Slack 봇)는 build_agent()만
호출해 항상 최신 에이전트를 받습니다.

지금 버전: 모델과 표준 도구.
"""

from __future__ import annotations

from langchain.agents import create_agent

from select_model import load_model
from tools import DEFAULT_TOOLS

DEFAULT_SYSTEM_PROMPT = (
    '당신은 유용한 AI 어시스턴트입니다. '
    '필요하면 도구를 사용하고, 도구를 실행하기 전 중간 과정을 간략히 설명하세요.'
)


def build_agent(model=None, tools=None, system_prompt=None, checkpointer=None):
    """현재까지 구성된 에이전트를 만들어 반환합니다.

    Args:
        model: chat model 또는 모델 이름. 생략 시 기본 모델을 로드합니다.
        tools: 도구 목록. 생략 시 tools.py의 DEFAULT_TOOLS를 씁니다.
        system_prompt: 시스템 프롬프트. 생략 시 기본값을 씁니다.
        checkpointer: 멀티턴 대화를 위한 체크포인터.
    """
    if model is None:
        model = load_model()
    if tools is None:
        tools = DEFAULT_TOOLS
    if system_prompt is None:
        system_prompt = DEFAULT_SYSTEM_PROMPT
    return create_agent(
        model,
        tools=tools,
        system_prompt=system_prompt,
        checkpointer=checkpointer,
    )


`build_agent.py`는 다음과 같이 실행합니다.

In [ ]:
from build_agent import build_agent

agent = build_agent()
result = await stream_with_markdown(agent, '오늘 날짜 알려주고, 올해가 며칠 지났는지 계산해줘')

---
## [실습] 나만의 ReAct Agent 만들기

아래 조건에 맞는 에이전트를 만들어보세요:

1. 새로운 도구를 1개 이상 추가
2. 에이전트에 적합한 시스템 프롬프트 작성
3. 여러 도구를 조합해야 하는 질문으로 테스트

In [ ]:
# TODO: 새로운 도구를 정의하세요
# @tool
# def my_custom_tool(...) -> str:
#     """도구 설명"""
#     pass

from langchain.tools import tool
from langchain.messages import HumanMessage

@tool
def exchange_rate_calculator(amount: float, from_currency: str, to_currency: str) -> str:
    """금액과 통화 코드를 입력받아 다른 통화로 환전한 예상 금액을 계산합니다.

    지원 통화: KRW, USD, EUR, JPY

    Args:
        amount: 환전할 금액
        from_currency: 원래 통화 코드. 예: KRW, USD, EUR, JPY
        to_currency: 바꿀 통화 코드. 예: KRW, USD, EUR, JPY
    """
    rates_to_krw = {
        "KRW": 1.0,
        "USD": 1380.0,
        "EUR": 1600.0,
        "JPY": 9.5,
    }

    source = from_currency.upper()
    target = to_currency.upper()

    if source not in rates_to_krw or target not in rates_to_krw:
        supported = ", ".join(rates_to_krw.keys())
        return f"지원하지 않는 통화입니다. 지원 통화: {supported}"

    amount_in_krw = amount * rates_to_krw[source]
    converted = amount_in_krw / rates_to_krw[target]

    return f"{amount:,.2f} {source} = 약 {converted:,.2f} {target} 입니다."


In [ ]:
# TODO: 에이전트를 생성하세요
# my_agent = create_agent(
#     'gpt-5.2',
#     tools=[...],
#     system_prompt='...'
# )

tools = DEFAULT_TOOLS + [exchange_rate_calculator]

my_agent = create_agent(
    'gpt-5.2',
    tools=tools,
    system_prompt='도구를 실행하기 전, 중간 과정을 자세히 설명하세요. 환율은 반드시 오늘날짜로 검색해서 찾아주고, 환율에 소수점 표시가 필요한 경우 소수점 둘째자리까지만 표현을 해주고 연산이 필요하면 소수점 둘째자리에서 반올림 해줘.'
)

In [ ]:
# TODO: 에이전트를 실행하세요 (또는 markdown stream하세요)
# result = my_agent.invoke({'messages': [{'role': 'user', 'content': '...'}]})
# print(result['messages'][-1].text)

result = my_agent.invoke({'messages': [{'role': 'user', 'content': '오늘날짜에 맞는 달러 환율과 엔화 환율을 검색해서 10달러를 원화로 바꾸면 얼마인지 알려주고, 원화를 다시 엔화로 환산 시 얼마나 되는지 알려줘.'}]})
print(result['messages'])
# print(result['messages'][-1].text)

In [ ]:
rprint(result['messages'])

---
## 정리

이번 실습에서 학습한 내용을 정리합니다.

- ReAct 패턴: LLM이 생각(Reasoning)과 행동(Acting)을 반복하며 문제를 해결하는 패턴
- create_agent: LangChain 1.x의 표준 에이전트 생성 API. 내부적으로 LangGraph 기반 그래프 구성
- 표준 도구: web_search(검색), calculator(계산), get_current_datetime(날짜/시간), text_analyzer(텍스트 분석), save_note/get_notes(메모)를 import해 활용
- System Prompt: 에이전트의 성격과 행동 방식을 결정하는 핵심 요소
- Checkpointer: `InMemorySaver`를 사용하면 `thread_id` 기반의 멀티턴 대화가 가능
- 메시지 흐름: `HumanMessage -> AIMessage(tool_calls) -> ToolMessage -> AIMessage(응답)` 패턴이 반복